In [ ]:
import pandas as pd
import sqlite3

In [13]:
import pandas as pd

plans = pd.read_csv("../plans.csv")
claims = pd.read_csv("../claims.csv")

plans.head()

,plan_id,plan_name,monthly_premium,annual_deductible,copay_pct,coverage_type,network_tier
0,P101,Gold PPO,500,2000,10,PPO,Gold
1,P102,Silver HMO,300,1500,20,HMO,Silver
2,P103,Bronze HMO,150,1000,30,HMO,Bronze


In [14]:
plans.info()
claims.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   plan_id            3 non-null      object
 1   plan_name          3 non-null      object
 2   monthly_premium    3 non-null      int64 
 3   annual_deductible  3 non-null      int64 
 4   copay_pct          3 non-null      int64 
 5   coverage_type      3 non-null      object
 6   network_tier       3 non-null      object
dtypes: int64(3), object(4)
memory usage: 300.0+ bytes
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   claim_id      5 non-null      object
 1   member_id     5 non-null      object
 2   plan_id       5 non-null      object
 3   procedure     5 non-null      object
 4   claim_amount  5 non-null      int64 
 5   status        5 non-nul

In [15]:
plans = plans.drop_duplicates()
claims = claims.drop_duplicates()

claims["date_filed"] = pd.to_datetime(claims["date_filed"])

In [18]:
import pandas as pd
import sqlite3

In [19]:
conn = sqlite3.connect("../coverage.db")

plans.to_sql("plans", conn, index=False, if_exists="replace")
claims.to_sql("claims", conn, index=False, if_exists="replace")

5

In [20]:
pd.read_sql("SELECT * FROM plans", conn)

,plan_id,plan_name,monthly_premium,annual_deductible,copay_pct,coverage_type,network_tier
0,P101,Gold PPO,500,2000,10,PPO,Gold
1,P102,Silver HMO,300,1500,20,HMO,Silver
2,P103,Bronze HMO,150,1000,30,HMO,Bronze


In [21]:
pd.read_sql("SELECT * FROM claims", conn)


,claim_id,member_id,plan_id,procedure,claim_amount,status,date_filed
0,C1001,M1001,P101,X-ray,250,Pending,2023-04-01 00:00:00
1,C1002,M1001,P101,Surgery,1200,Approved,2023-03-15 00:00:00
2,C1003,M1002,P102,X-ray,150,Denied,2023-04-05 00:00:00
3,C1004,M1002,P102,Surgery,900,Approved,2023-03-20 00:00:00
4,C1005,M1003,P103,X-ray,50,Pending,2023-04-10 00:00:00


In [22]:
pd.read_sql("""
SELECT annual_deductible
FROM plans
WHERE plan_name='Gold PPO'
""",conn)

,annual_deductible
0,2000


In [23]:
pd.read_sql("""
SELECT COUNT(*) AS PendingClaims
FROM claims
WHERE member_id='M1001'
AND status='Pending'
""",conn)

,PendingClaims
0,1


In [24]:
pd.read_sql("""
SELECT plan_name
FROM plans
WHERE monthly_premium<400
""",conn)

,plan_name
0,Silver HMO
1,Bronze HMO


In [25]:
pd.read_sql("""
SELECT member_id,
plan_name,
claim_amount,
status
FROM claims
JOIN plans
ON claims.plan_id=plans.plan_id
""",conn)

,member_id,plan_name,claim_amount,status
0,M1001,Gold PPO,250,Pending
1,M1001,Gold PPO,1200,Approved
2,M1002,Silver HMO,150,Denied
3,M1002,Silver HMO,900,Approved
4,M1003,Bronze HMO,50,Pending


In [26]:
pd.read_sql("""
SELECT procedure,
COUNT(*) AS total
FROM claims
GROUP BY procedure
ORDER BY total DESC
""",conn)

,procedure,total
0,X-ray,3
1,Surgery,2


In [27]:
conn.close()